Global vs. Local Batch Size

Students often mistakenly set the batch size in their script to 64, thinking the whole cluster will process 64 images. In reality, if they have 4 GPUs, they are accidentally processing a global batch of 256! This code shows them how to dynamically calculate the correct batch size based on the hardware available.

In [1]:
import tensorflow as tf

# 1. Initialize your strategy
strategy = tf.distribute.MultiWorkerMirroredStrategy()

# 2. Define what ONE single GPU can handle without running out of VRAM
per_replica_batch_size = 64

# 3. Calculate the Global Batch Size dynamically
num_workers = strategy.num_replicas_in_sync
global_batch_size = per_replica_batch_size * num_workers

print(f"Number of GPUs: {num_workers}")
print(f"Local Batch Size: {per_replica_batch_size}")
print(f"Global Batch Size: {global_batch_size}")

2026-03-01 20:18:02.979880: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-01 20:18:03.042530: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-01 20:18:04.586594: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


INFO:tensorflow:Using MirroredStrategy with devices ('/device:GPU:0', '/device:GPU:1')
INFO:tensorflow:Single-worker MultiWorkerMirroredStrategy with local_devices = ('/device:GPU:0', '/device:GPU:1'), communication = CommunicationImplementation.AUTO
Number of GPUs: 2
Local Batch Size: 64
Global Batch Size: 128


I0000 00:00:1772417885.276510  809864 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 687 MB memory:  -> device: 0, name: Quadro RTX 5000, pci bus id: 0000:17:00.0, compute capability: 7.5
I0000 00:00:1772417885.277819  809864 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 688 MB memory:  -> device: 1, name: Quadro RTX 5000, pci bus id: 0000:b3:00.0, compute capability: 7.5


Manual Sharding (Dividing the Data)

If you don't shard, Worker 0 and Worker 1 will both read the exact same TFRecord file and compute the exact same gradients. The shard() method is the mathematical trick that forces each worker to skip the data meant for the others.

In [5]:
import tensorflow as tf
import os, json

# 1. Load the full 60,000 image dataset on both nodes
(x_train, y_train), _ = tf.keras.datasets.mnist.load_data()
dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train))

# 2. Get the specific Node's identity
tf_config = json.loads(os.environ.get('TF_CONFIG', '{}'))
node_index = tf_config.get('task', {}).get('index', 0)
total_nodes = 2

# 3. Print size BEFORE sharding
initial_size = tf.data.experimental.cardinality(dataset).numpy()
print(f"Node {node_index} | Size before sharding: {initial_size} images")

# 4. Apply the Shard
dataset = dataset.shard(num_shards=total_nodes, index=node_index)

# 5. Print size AFTER sharding
sharded_size = tf.data.experimental.cardinality(dataset).numpy()
print(f"Node {node_index} | Size AFTER sharding: {sharded_size} images")

# 6. NOW you can safely apply repeat() and batch()
dataset = dataset.repeat().batch(64, drop_remainder=True)

Node 0 | Size before sharding: 60000 images
Node 0 | Size AFTER sharding: 30000 images


Ingesting the Data with from_tensor_slices()

Before we can shuffle, batch, or distribute our data across the network, we must convert it from static NumPy arrays sitting in the CPU's RAM into a dynamic TensorFlow pipeline.

The tf.data.Dataset.from_tensor_slices() function takes block of 60,000 images and labels and "slices" them along their first dimension. This transforms a giant, unmanageable block of memory into a streamable conveyor belt of individual (image, label) pairs that GPUs can safely ingest one batch at a time without crashing.

In [ ]:
import os
import json
import tensorflow as tf

def build_distributed_mnist_pipeline(x_train, y_train, global_batch_size):
    # --- 1. Hardware & Node Discovery ---
    tf_config = json.loads(os.environ.get('TF_CONFIG', '{}'))
    node_index = tf_config.get('task', {}).get('index', 0)
    total_nodes = len(tf_config.get('cluster', {}).get('worker', ['dummy_worker']))
    local_gpus = len(tf.config.list_physical_devices('GPU'))
    
    print(f"NODE {node_index} PIPELINE DASHBOARD")
    print(f"Local GPUs Detected: {local_gpus}")
    print(f"Total Cluster Nodes: {total_nodes}")
    
    # --- 2. Initial Dataset Creation ---
    dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train))
    initial_size = tf.data.experimental.cardinality(dataset).numpy()
    print(f"Global Data Size:    {initial_size} images")    

    # --- 4. Pipeline Optimization ---
    dataset = dataset.cache()
    dataset = dataset.shuffle(buffer_size=10000)
    dataset = dataset.repeat()
    dataset = dataset.batch(global_batch_size, drop_remainder=True)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    
    return dataset

train_dist_dataset = build_distributed_mnist_pipeline(x_train, y_train, global_batch_size=128)

NODE 0 PIPELINE DASHBOARD
Local GPUs Detected: 2
Total Cluster Nodes: 1
Global Data Size:    60000 images


: 

The Modern "Auto-Shard" Shortcut

TensorFlow will attempt to apply the sharding math automatically!

In [3]:
import os
import numpy as np

# 1. Force Legacy Keras 2
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import tensorflow as tf

def create_robust_dataset(features, labels, global_batch_size, num_gpus):
    """Builds a standard dataset for single-node multi-GPU training."""
    dataset = tf.data.Dataset.from_tensor_slices((features, labels))
    
    # --- DASHBOARD LOGIC ---
    global_size = tf.data.experimental.cardinality(dataset).numpy()
    local_batch_size = global_batch_size // num_gpus
    
    print(f"\nAUTO-SHARDING DASHBOARD (SINGLE NODE)")
    print(f"Local GPUs Detected:    {num_gpus}")
    print(f"Global Data Size:       {global_size} images")
    print(f"Global Batch Size:      {global_batch_size} (Pulled from CPU RAM)")
    print(f"Per-GPU Batch Size:     {local_batch_size} (Sent to each GPU)")
    print(f"Note: MirroredStrategy will automatically slice every")
    print(f"      global batch across your {num_gpus} local GPUs!\n")

    # --- PIPELINE LOGIC ---
    dataset = dataset.cache()
    dataset = dataset.shuffle(buffer_size=10000)
    dataset = dataset.repeat()
    # We batch using the GLOBAL size. The strategy splits it later!
    dataset = dataset.batch(global_batch_size, drop_remainder=True)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    
    return dataset

# 2. Initialize Strategy
# MirroredStrategy automatically detects GPUs
strategy = tf.distribute.MirroredStrategy()

# 3. Calculate Batch Sizes
num_gpus = strategy.num_replicas_in_sync
per_replica_batch_size = 64
global_batch_size = per_replica_batch_size * num_gpus

# 4. Load MNIST data
print("Loading MNIST dataset...")
(x_train, y_train), _ = tf.keras.datasets.mnist.load_data()
x_train = np.expand_dims(x_train.astype(np.float32) / 255.0, -1)
y_train = y_train.astype(np.float32)

# 5. Build a normal dataset
normal_dataset = create_robust_dataset(x_train, y_train, global_batch_size, num_gpus)

# 6. Let the Strategy automatically shard it and route it to the GPUs
distributed_dataset = strategy.experimental_distribute_dataset(normal_dataset)

print(f"Successfully created Auto-Sharded Distributed Dataset!")
print(f"Type: {type(distributed_dataset)}")


INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
Loading MNIST dataset...

AUTO-SHARDING DASHBOARD (SINGLE NODE)
Local GPUs Detected:    2
Global Data Size:       60000 images
Global Batch Size:      128 (Pulled from CPU RAM)
Per-GPU Batch Size:     64 (Sent to each GPU)
Note: MirroredStrategy will automatically slice every
      global batch across your 2 local GPUs!

Successfully created Auto-Sharded Distributed Dataset!
Type: <class 'tensorflow.python.distribute.input_lib.DistributedDataset'>
